# Sabueso retrieval and entity resolution
Offline public UniProt fixtures (CC BY 4.0; see experiments/fixtures/sabueso_uniprot/NOTICE.md). Start once, call declared boundaries across cells, stop once. Sabueso owns Cards, decisions and acquisition traces; Recorda owns operation receipts. An exact ConnectorError adapter retains native exception trace references on failed operations. This experiment does not capture arbitrary calls or replay a workflow.


In [ ]:
import sys
import uuid
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "experiments"))
import recorda
from inspect_sabueso_trial import inspect_trial
from sabueso.core.errors import ConnectorError
from sabueso.resolver import EntityQuery, EntityResolver
from sabueso_consumer import (
    SabuesoArtifacts,
    fixture_client,
    get_entry,
    resolve,
    resolve_retrieved_entry,
)

destination = Path("artifacts") / ("sabueso-" + uuid.uuid4().hex)
destination.mkdir(parents=True)
artifacts = SabuesoArtifacts(destination / "native")
resolver = EntityResolver(fixture_client())
failing_client = fixture_client(failing=True)
failure_resolver = EntityResolver(failing_client)
print(sys.executable)

In [ ]:
handle = recorda.start(
    "sabueso-notebook",
    path=destination / "session.jsonl",
    reference_adapters=artifacts.adapters,
    gaps=[
        "unwrapped source helpers",
        "unregistered exception provenance is omitted",
        "no MOLI context or complete capture",
    ],
)

In [ ]:
envelope = get_entry("P60174", client=fixture_client())
assert envelope["record"]["primaryAccession"] == "P60174"

In [ ]:
card, resolution = resolve_retrieved_entry(envelope, resolver)
assert resolution.status == "resolved"
print(card.pinned_ref())

In [ ]:
ambiguous = resolve(EntityQuery(identifier="P00938"), resolver=resolver)
selected = resolve(EntityQuery(identifier="P00938", organism=9606), resolver=resolver)
assert ambiguous[0] is None and ambiguous[1].status == "ambiguous"
assert selected[1].entity_ref == "sabueso:protein:uniprot:P60174"
assert selected[1].alternatives[0]["entity_ref"] == "sabueso:protein:uniprot:P60175"

In [ ]:
returned_error = resolve("P12345", resolver=failure_resolver)
assert returned_error[0] is None and returned_error[1].status == "error"
# A returned semantic error is a completed call, recorded as succeeded.

In [ ]:
try:
    get_entry("P12345", client=failing_client)
except ConnectorError:
    pass
# Native trace retention is performed by the registered exact-type adapter.
# P12345 is an injected fixture failure, not a claim about live UniProt.

In [ ]:
record = handle.stop()
assert record.status == "failed" and len(record.operations) == 7
assert record.operations[-1]["exception"]["reference"]["owner"] == "sabueso"
artifacts.finish(journal="session.jsonl")
report = inspect_trial(destination)
assert report["network_attempts"] == 0
assert report["semantic_statuses"] == ["resolved", "resolved", "ambiguous", "resolved", "error"]
[(op["name"], op["execution_status"], op.get("semantic_status")) for op in report["operations"]]